# Vorbereitung

## Importiere Packages
Installiere für dieses Notebook notwendige Packages.

In [1]:
import subprocess

# FILENAME_REQUIREMENTS = 'requirements.txt'

try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:

    # Install the package 'thefuzz'.
    subprocess.run(['pip', 'install', 'thefuzz'], check=True)

# else:

    # Install all packages listed in local available requirements.txt.
    # Isubprocess.run(['pip', 'install', '-r', FILENAME_REQUIREMENTS], check=True)

- Importiere für dieses Skript weitere notwendige Funktionen.

In [2]:
import zipfile
import os
from lxml import etree as et
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from pyproj import Transformer
from ipyleaflet import Map, WMSLayer, Circle, Popup
import ipywidgets as widgets
from thefuzz import fuzz, process
import re
from rapidfuzz import fuzz

 - Packages für geobasierte Visualisierungen

In [3]:

# 1. Paket im Terminal installieren (nur in Jupyter Notebooks möglich!)
%pip install shapely

# Import packages.
import pandas as pd
from pyproj import Transformer
from ipyleaflet import Map, WMSLayer, Circle, Popup, Heatmap, LayerGroup, LegendControl
import ipywidgets as widgets
import time

# shapely ebenfalls importieren
import shapely

from ipyleaflet import Polygon



Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Definiere Arbeitsverzeichnis für Google Colab
Definiere Arbeitsverzeichnis in Google Colab zur Speicherung Daten und Ergebnisse.

In [4]:
try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
  # Define the working directory in google colab.
  WORKING_DIRECTORY = '/content/drive/My Drive/Colab Notebooks/'

  # Mount personal google drive.
  drive.mount('/content/drive')

## Bereite Datengrundlage auf
Lade und entpacke für die Suche verwendete Datengrundlage.

- Dieser Code bedient zwei Szenarien: 

    - Falls mit Colab gearbeitet wird, sollte das Laden und Entpacken direkt über den Link aud GitHub funktioneriern.

    - Falls lokal gearbeitet wird, muss die Datei separat heruntergeldan und in die lokale Umgebung zusätzlich eingepflegt werden

In [5]:
FILENAME_DATA = 'hgb_corpus_24_07_26_inline_full_updated.zip'
URL_DATA = 'https://github.com/history-unibas/FS2024-Research-Seminar-Demo-Notebook/raw/main/hgb_corpus_24_07_26_inline_full_updated.zip'


if in_colab:

    if not os.path.exists(WORKING_DIRECTORY + FILENAME_DATA):

      # Download the data in working directory.
      subprocess.run(['wget', URL_DATA, '-O', WORKING_DIRECTORY + FILENAME_DATA], check=True)

      # Unzip the file.
      with zipfile.ZipFile(WORKING_DIRECTORY + FILENAME_DATA, 'r') as myzip:
          myzip.extractall(WORKING_DIRECTORY)

    # Update the filepath.
    FILENAME_DATA = WORKING_DIRECTORY + os.path.splitext(FILENAME_DATA)[0] + '.xml'

else:

    # Unzip the file.
    with zipfile.ZipFile(FILENAME_DATA, 'r') as myzip:
        myzip.extractall('.')

    # Update the filepath.
    FILENAME_DATA = os.path.splitext(FILENAME_DATA)[0] + '.xml'

# Element-Orientierte Suche über XPath-abfragen


In [6]:
# Define Search Parameters
YEAR_MIN = 1400
YEAR_MAX = 1530

# Your XPATH Search
# zu erst  wird auf die  Gesamtmenge der  Orgas reduziert, damit wir innerhalb davon weitere Reduktionen (rel) und Filter vornehmen können 


XPATH_EXP = "./Body//Descriptor[@desc_type='tax']/Reference[@entity_type='org']"

#XPATH_EXP = "./Body//Reference[@entity_type = 'org' and @entity_subtype = 'rel']"


In [7]:
# Load xml file.
context = et.iterparse(FILENAME_DATA, events=('start', 'end'))

# Collector for our results
results = []

# Iterate over each element.
collection_element = None

for event, elem in context:
    if event == 'end' and elem.tag == 'Document':

        # Use XPATH to filter document year
        if not elem.xpath(f"./Header[not(@year < {YEAR_MIN} or @year > {YEAR_MAX})]"):
            continue

        # Add all found xml-elements to our results-list
        results.extend(elem.xpath(XPATH_EXP))
        #results.extend(elem.xpath(XPATH_EXP_2))

print(f"Found {len(results)} elements that matched the xpath criteria!")

Found 14113 elements that matched the xpath criteria!


# Umwandlung zur Tabelle
Haben wir unsere Elemente gefunden und mögliche Filtern verwendet, können wir die Resultate in ein `Dataframe` umwandeln, quasi eine tabellarische Struktur. Diese kennt ihr schon aus den früheren Demo-Notebooks und könnt dann damit weiterarbeiten.

Denkt daran, dass `result` in diesem Schritt immer noch ein XML-Element beinhaltet, das bedeutet, ihr könnt in eurer Tabelle auch Informationen z.B. über den Parent des Elements verzeichnen.

In [8]:



def clean_whitespaces(text):
    """Replaces all whitespaces with single space characters."""
    return re.sub("\s+", " ", text)


datacollector = []

for result in results:
    data = {}

    # Metadaten zum Dossier
    collection_element = result.xpath("ancestor::Collection")[0]
    data["coll_id"] = collection_element.get("id")
    data["coll_house"] = collection_element.get("house")
    data["coll_x"] = collection_element.get("coord_x")
    data["coll_y"] = collection_element.get("coord_y")

    # Metadaten zum Dokument
    document_element = result.xpath("ancestor::Document")[0]
    data["doc_id"] = document_element.get("id")
    header = document_element.find("Header")
    data["doc_pageNum"] = header.get("pages")
    data["doc_date"] = int(header.get("year"))  # direkt als int
    data["doc_fulltext"] = header.get("text")
    data["doc_image"] = header.get("imagelinks")

    # Suchresultat
    data["res_full"] = clean_whitespaces("".join(result.xpath(".//text()")))
    data["res_head"] = clean_whitespaces("".join(result.find("Head").xpath(".//text()")) if result.find("Head") is not None else "")

    # weiteres
    data["ner_confidence"] = result.get("confidence")

    # Kategorie
    data["res_cat"] = result.get("entity_subtype")

    # Normalisierte Version der Organisation
    data["res_norm"] = result.get("normalization")
    data["res_norm_confidence"] = result.get("norm_confidence")

    datacollector.append(data)

result_df = pd.DataFrame.from_dict(datacollector)

display(result_df)


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/var/folders/n4/t9gk6y5969s0ptnyp8q2q_th0000gq/T/ipykernel_14167/713165822.py:3: SyntaxWarning: invalid escape sequence '\s'
  return re.sub("\s+", " ", text)


,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence
0,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,47bc059d-7518-493d-b496-963323e417e0_20240705,11,1439,Peter Hans Wittenhein der segkler u. Angnes si...,https://files.transkribus.eu/Get?id=YEUCFVOURA...,dem Closter zen Barfüssen,zen Barfüssen,0.9982364475727081,rel,Barfüsserkloster,1.0
1,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,47bc059d-7518-493d-b496-963323e417e0_20240705,11,1439,Peter Hans Wittenhein der segkler u. Angnes si...,https://files.transkribus.eu/Get?id=YEUCFVOURA...,zes . Lienh .,zes . Lienh .,0.8069755584001541,rel,Kirche St. Leonhard,0.9735923609725222
2,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,dem Spital,Spital,0.9983475208282471,soc,Heiliggeist-Spital,1.0
3,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,das gotshus s . Lienh .,gotshus s . Lienh .,0.9837589462598165,rel,Kirche St. Leonhard,1.0
4,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,der Carthuser closter,Carthuser closter,0.9948348005612692,rel,Kartause,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14108,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,2d9cb1c5-a427-45b6-8e1d-6fa8723aac9e_20240705,5,1436,Item git ze kouffen Gredlin Kathe Henß\nlin Mu...,https://files.transkribus.eu/Get?id=QMLKONQJEQ...,den frowen zu sant Cla - ren,frowen zu sant Cla - ren,0.9792102490152631,rel,Kloster St. Klara,0.9552494690236285
14109,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,4cbbada6-1de3-40a3-b483-691e9b1a31e7_20240705,12,1482,Item git zu koffend Hans Grünenstein er¬\nmüll...,https://files.transkribus.eu/Get?id=KHCVFRVBBO...,dem gotz huß zu sant Claren,gotz huß zu sant Claren,0.9751211106777191,rel,Kloster St. Klara,0.9662814089548328
14110,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,a22a448f-e030-4fbb-b4a9-880da5bfddd2_20240705,13,1482,Item gend zu koffend Hans Fierstein der\nwaffe...,https://files.transkribus.eu/Get?id=SQIRCESMVX...,dem gotzhuß zu sant Claren,gotzhuß zu sant Claren,0.986992347240448,rel,Kloster St. Klara,1.0
14111,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,6d644da3-4ba4-42a4-a496-df4c6948a512_20240705,18,1507,Item da gibt zu kouffen Müllerli burgern\nze B...,https://files.transkribus.eu/Get?id=NZTRJZZGHR...,den andechtigen frowen zu sant Claren,frowen zu sant Claren,0.9829913675785065,rel,Kloster St. Klara,1.0


In [22]:
result_df

,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence
0,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,47bc059d-7518-493d-b496-963323e417e0_20240705,11,1439,Peter Hans Wittenhein der segkler u. Angnes si...,https://files.transkribus.eu/Get?id=YEUCFVOURA...,dem Closter zen Barfüssen,zen Barfüssen,0.9982364475727081,rel,Barfüsserkloster,1.0
1,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,47bc059d-7518-493d-b496-963323e417e0_20240705,11,1439,Peter Hans Wittenhein der segkler u. Angnes si...,https://files.transkribus.eu/Get?id=YEUCFVOURA...,zes . Lienh .,zes . Lienh .,0.8069755584001541,rel,Kirche St. Leonhard,0.9735923609725222
2,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,dem Spital,Spital,0.9983475208282471,soc,Heiliggeist-Spital,1.0
3,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,das gotshus s . Lienh .,gotshus s . Lienh .,0.9837589462598165,rel,Kirche St. Leonhard,1.0
4,HGB_1_024_012,Barfüsserplatz 1,2611325.2930335696,1267174.6259057096,98af92a3-b2dd-489d-aee3-90d0ac743d16_20240705,15,1457,Die Ellenden Herberg verkauft an Michel Vische...,https://files.transkribus.eu/Get?id=TRXHVGNWRG...,der Carthuser closter,Carthuser closter,0.9948348005612692,rel,Kartause,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14108,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,2d9cb1c5-a427-45b6-8e1d-6fa8723aac9e_20240705,5,1436,Item git ze kouffen Gredlin Kathe Henß\nlin Mu...,https://files.transkribus.eu/Get?id=QMLKONQJEQ...,den frowen zu sant Cla - ren,frowen zu sant Cla - ren,0.9792102490152631,rel,Kloster St. Klara,0.9552494690236285
14109,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,4cbbada6-1de3-40a3-b483-691e9b1a31e7_20240705,12,1482,Item git zu koffend Hans Grünenstein er¬\nmüll...,https://files.transkribus.eu/Get?id=KHCVFRVBBO...,dem gotz huß zu sant Claren,gotz huß zu sant Claren,0.9751211106777191,rel,Kloster St. Klara,0.9662814089548328
14110,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,a22a448f-e030-4fbb-b4a9-880da5bfddd2_20240705,13,1482,Item gend zu koffend Hans Fierstein der\nwaffe...,https://files.transkribus.eu/Get?id=SQIRCESMVX...,dem gotzhuß zu sant Claren,gotzhuß zu sant Claren,0.986992347240448,rel,Kloster St. Klara,1.0
14111,HGB_1_017_018,Badergässlein 14,2611501.827341824,1267907.0956957322,6d644da3-4ba4-42a4-a496-df4c6948a512_20240705,18,1507,Item da gibt zu kouffen Müllerli burgern\nze B...,https://files.transkribus.eu/Get?id=NZTRJZZGHR...,den andechtigen frowen zu sant Claren,frowen zu sant Claren,0.9829913675785065,rel,Kloster St. Klara,1.0


# Fuzzysuche nach bestimmten Begriffen oder längeren variierenden Strings

In [9]:

# bezieht sich auf `result_df`
# wir suchen in der Spalte "doc_fulltext" oder "res_full"

keyword = "Frowen uff Burg"
threshold = 80

# Neue Spalte, die True/False angibt, ob ein Fuzzy-Match über Threshold gefunden wurde
result_df["fuzzy_match"] = result_df["doc_fulltext"].apply(
    lambda text: fuzz.partial_ratio(keyword.lower(), text.lower()) >= threshold if isinstance(text, str) else False
)

# Zeige nur Zeilen mit Match
matched_df = result_df[result_df["fuzzy_match"]]

display(matched_df)

,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence,fuzzy_match
350,HGB_1_058_021,Freie Strasse 3,2611284.8859377317,1267445.7386366418,9b3b19c0-cb52-44e4-b8aa-b6329ca66e7e_20240705,11,1499,"Pentelin Zschüpp der Messerschmid, u seine Fra...",https://files.transkribus.eu/Get?id=FHKMGYYSLQ...,der Presenz uff Burg,Presenz,0.9681175500154495,oth,Präsenz,1.0,True
489,HGB_1_058_053,Freie Strasse Theil von 51,2611408.880195439,1267278.9871398073,f25f1ecd-14e9-45f1-9e9a-e396ac89b703_20240705,8,1408,Heman Slosser der Messerschmidt Anna eins uxor...,https://files.transkribus.eu/Get?id=WDUZXYBRWA...,altaris sanctorum,altaris sanctorum,0.6100419163703918,,no_label,1.0,True
490,HGB_1_058_053,Freie Strasse Theil von 51,2611408.880195439,1267278.9871398073,f25f1ecd-14e9-45f1-9e9a-e396ac89b703_20240705,8,1408,Heman Slosser der Messerschmidt Anna eins uxor...,https://files.transkribus.eu/Get?id=WDUZXYBRWA...,den Thurnherren uff Burg,Thurnherren,0.9772415459156036,rel,Domstift,1.0,True
671,HGB_1_058_070,Freie Strasse 79,2611452.672265297,1267202.46676235,4c825b09-2893-47da-a242-03dee97e7e7c_20240705,16,1416,Item do gab ze kouffend frow Anna Griein\nrelt...,https://files.transkribus.eu/Get?id=TJMGXEGIUH...,einer pfrund sant Vincenigen altares,pfrund sant Vincenigen altares,0.8457573890686035,,no_label,1.0,True
672,HGB_1_058_070,Freie Strasse 79,2611452.672265297,1267202.46676235,4c825b09-2893-47da-a242-03dee97e7e7c_20240705,16,1416,Item do gab ze kouffend frow Anna Griein\nrelt...,https://files.transkribus.eu/Get?id=TJMGXEGIUH...,den Barfüssen,Barfüssen,0.986118495464325,rel,Barfüsserkloster,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13622,HGB_1_218_028,Streitgasse 20,2611399.294296451,1267178.5328754832,6079bd15-383a-4cca-ba2f-2ee62c66cf6a_20240705,34,1491,Schultheissen Urkunde.\nLienhart Muge der Mure...,https://files.transkribus.eu/Get?id=IDWORCKRYS...,uff Burg zu Basel,uff Burg,0.8522274345159531,,no_label,1.0,True
13623,HGB_1_218_028,Streitgasse 20,2611399.294296451,1267178.5328754832,6079bd15-383a-4cca-ba2f-2ee62c66cf6a_20240705,34,1491,Schultheissen Urkunde.\nLienhart Muge der Mure...,https://files.transkribus.eu/Get?id=IDWORCKRYS...,b . Frowen Buw,b . Frowen Buw,0.8733343482017517,rel,Münsterbau,1.0,True
13624,HGB_1_218_028,Streitgasse 20,2611399.294296451,1267178.5328754832,6079bd15-383a-4cca-ba2f-2ee62c66cf6a_20240705,34,1491,Schultheissen Urkunde.\nLienhart Muge der Mure...,https://files.transkribus.eu/Get?id=IDWORCKRYS...,den Herren zu St . Peter,Herren zu St . Peter,0.9798332750797272,rel,Kirche St. Peter,1.0,True
13627,HGB_1_218_028,Streitgasse 20,2611399.294296451,1267178.5328754832,df9cfb28-e905-466b-8b90-66dad4bdf75c_20240705,36,1492,"Lienhart Müge der Mürer u seine Frau Margreth,...",https://files.transkribus.eu/Get?id=KTBBSRDHIA...,u . lefrauen uff Burg,u . lefrauen,0.9393517971038818,,no_label,0.9866655446553421,True


# Institutionsspezifische Suche

### Ausschliesslich den "String" mit der gewünschtten Institution austausschen

In [10]:
FILTER_INSTITUTION = "Kloster Klingental"

In [11]:
klingi_df = result_df[result_df["res_norm"] == FILTER_INSTITUTION]

klingi_df


,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence,fuzzy_match
64,HGB_1_024_040,Barfüsserplatz 15,2611302.138354082,1267114.9650077126,668dfb9d-8780-4d4a-a39b-4be080ecbfe2_20240705,14,1492,Hanns Rosenfeld der Wachtmeister verkauft an L...,https://files.transkribus.eu/Get?id=GTBFRSLYCP...,Clingental,Clingental,0.9173821210861206,rel,Kloster Klingental,1.0,False
159,HGB_1_024_066,Barfüsserplatz 29,2611333.3941644696,1267149.2194807678,ab16bc69-171e-4d91-81da-f43ccb09db9a_20240705,23,1523,"Melchior Lompar der Miner, u. seine Frau Ursul...",https://files.transkribus.eu/Get?id=JQFGPLGQSS...,einer Closterfrauen zu Clingental,Closterfrauen zu Clingental,0.88949915766716,rel,Kloster Klingental,1.0,False
252,HGB_1_024_115,Barfüsserplatz 28,2611327.614136818,1267149.1169899958,801760e2-13f7-4a7b-92ab-b92f55fd325a_20240705,19,1523,Melchior Lumpar der Kaurer u. seine Frau Ursul...,https://files.transkribus.eu/Get?id=BQPFCYPJFZ...,einer Closterfrauen zu Clingenthal,Closterfrauen zu Clingenthal,0.837795689702034,rel,Kloster Klingental,1.0,False
255,HGB_1_024_116,Barfüsserplatz 30,2611334.6649472862,1267157.820265856,5da065f0-d31c-4d3c-a489-04308819cd01_20240705,11,1454,Do gend ze kaufen Mr. Conrat Wagner der Lütenm...,https://files.transkribus.eu/Get?id=IJFCXNSZMK...,die frauen von Clingental,frauen von Clingental,0.9784636348485947,rel,Kloster Klingental,1.0,False
298,HGB_1_058_020,Freie Strasse 1,2611281.9863638678,1267451.4916674045,e5a59ebf-42f3-4e7c-8327-7c8061355f2a_20240705,22,1492,Schultheissen Urkunde.\nFrow Margreth von Buss...,https://files.transkribus.eu/Get?id=XPVYRMZPKN...,den Frowen zu Clingental,Frowen zu Clingental,0.8744285106658936,rel,Kloster Klingental,1.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14096,HGB_1_017_008,Badergässlein 3,2611478.9129142133,1267904.6579906673,ddb9dbd0-ab58-4226-b962-95737ec0f4d3_20240705,21,1464,"Frow Else Rocklins, wil. Oswald Stehelin des W...",https://files.transkribus.eu/Get?id=QDXKTZSOSR...,den Frowen zu Clingental,Frowen zu Clingental,0.9514731019735336,rel,Kloster Klingental,1.0,False
14098,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,6dbafa7c-75a6-45de-bb41-5fcad55ba20c_20240705,14,1416,Item het zu koffende zem Gred Frischers den\nh...,https://files.transkribus.eu/Get?id=ETMEFXMPSS...,den von Clingental,Clingental,0.9525679349899292,rel,Kloster Klingental,1.0,False
14099,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,d6ab6e15-5a06-4e7f-aade-17399fc78365_20240705,"20, 21",1426,Item git ze kouffende herr Schaler der Ziegler...,https://files.transkribus.eu/Get?id=OAPEIOSDJF...,dem obge frowen ze Klingental,frowen ze Klingental,0.9721857070922851,rel,Kloster Klingental,1.0,False
14100,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,903b0ee3-2d41-40b0-a25b-753076e3dc1b_20240705,22,1432,Item git ze kouffen Elsy relta quond Antz\nlin...,https://files.transkribus.eu/Get?id=NGZXVNXMCQ...,den frowen ze Klingental,frowen ze Klingental,0.9919654279947281,rel,Kloster Klingental,1.0,False


### Speichern der Tabelle in Colab oder in lokaler Umgebung

In [21]:

# 1. Google Drive Einbindung prüfen
try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

# 2. Pfad dynamisch anpassen
if in_colab:
    # Google Drive mounten
    drive.mount('/content/drive')
    # Zielordner in Google Colab / Drive
    base_path = '/content/drive/My Drive/Colab Notebooks/'
else:
    # Fallback für lokale Umgebung (aktuelles Arbeitsverzeichnis)
    base_path = "./"

# Dateiname festlegen
file_name = "klingi.csv"

# Vollständiger Pfad zur Zieldatei
file_path = os.path.join(base_path, file_name)

# DataFrame als CSV speichern (Beispiel-DataFrame 'klingi_df')
# Stelle sicher, dass klingi_df vorher im Code definiert wurde
klingi_df.to_csv(file_path, index=False)

print(f"Datei wurde gespeichert unter: {file_path}")


Datei wurde gespeichert unter: ./klingi.csv


## Geovisualisierung

# Transformiere die Koordinaten in WGS 84

In [13]:
# Transform the coordinates of the search results.
def transform_coords(x, y):
    transformer = Transformer.from_crs(2056, 4326)
    return transformer.transform(x, y)
klingi_df['lat'], klingi_df['lon'] = zip(*klingi_df.apply(
    lambda row: transform_coords(row['coll_x'], row['coll_y']), axis=1
    ))
klingi_df

/var/folders/n4/t9gk6y5969s0ptnyp8q2q_th0000gq/T/ipykernel_14167/1836728114.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  klingi_df['lat'], klingi_df['lon'] = zip(*klingi_df.apply(
/var/folders/n4/t9gk6y5969s0ptnyp8q2q_th0000gq/T/ipykernel_14167/1836728114.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  klingi_df['lat'], klingi_df['lon'] = zip(*klingi_df.apply(


,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence,fuzzy_match,lat,lon
64,HGB_1_024_040,Barfüsserplatz 15,2611302.138354082,1267114.9650077126,668dfb9d-8780-4d4a-a39b-4be080ecbfe2_20240705,14,1492,Hanns Rosenfeld der Wachtmeister verkauft an L...,https://files.transkribus.eu/Get?id=GTBFRSLYCP...,Clingental,Clingental,0.9173821210861206,rel,Kloster Klingental,1.0,False,47.554654,7.588776
159,HGB_1_024_066,Barfüsserplatz 29,2611333.3941644696,1267149.2194807678,ab16bc69-171e-4d91-81da-f43ccb09db9a_20240705,23,1523,"Melchior Lompar der Miner, u. seine Frau Ursul...",https://files.transkribus.eu/Get?id=JQFGPLGQSS...,einer Closterfrauen zu Clingental,Closterfrauen zu Clingental,0.88949915766716,rel,Kloster Klingental,1.0,False,47.554962,7.589192
252,HGB_1_024_115,Barfüsserplatz 28,2611327.614136818,1267149.1169899958,801760e2-13f7-4a7b-92ab-b92f55fd325a_20240705,19,1523,Melchior Lumpar der Kaurer u. seine Frau Ursul...,https://files.transkribus.eu/Get?id=BQPFCYPJFZ...,einer Closterfrauen zu Clingenthal,Closterfrauen zu Clingenthal,0.837795689702034,rel,Kloster Klingental,1.0,False,47.554961,7.589115
255,HGB_1_024_116,Barfüsserplatz 30,2611334.6649472862,1267157.820265856,5da065f0-d31c-4d3c-a489-04308819cd01_20240705,11,1454,Do gend ze kaufen Mr. Conrat Wagner der Lütenm...,https://files.transkribus.eu/Get?id=IJFCXNSZMK...,die frauen von Clingental,frauen von Clingental,0.9784636348485947,rel,Kloster Klingental,1.0,False,47.555039,7.589209
298,HGB_1_058_020,Freie Strasse 1,2611281.9863638678,1267451.4916674045,e5a59ebf-42f3-4e7c-8327-7c8061355f2a_20240705,22,1492,Schultheissen Urkunde.\nFrow Margreth von Buss...,https://files.transkribus.eu/Get?id=XPVYRMZPKN...,den Frowen zu Clingental,Frowen zu Clingental,0.8744285106658936,rel,Kloster Klingental,1.0,False,47.557681,7.588517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14096,HGB_1_017_008,Badergässlein 3,2611478.9129142133,1267904.6579906673,ddb9dbd0-ab58-4226-b962-95737ec0f4d3_20240705,21,1464,"Frow Else Rocklins, wil. Oswald Stehelin des W...",https://files.transkribus.eu/Get?id=QDXKTZSOSR...,den Frowen zu Clingental,Frowen zu Clingental,0.9514731019735336,rel,Kloster Klingental,1.0,False,47.561754,7.591145
14098,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,6dbafa7c-75a6-45de-bb41-5fcad55ba20c_20240705,14,1416,Item het zu koffende zem Gred Frischers den\nh...,https://files.transkribus.eu/Get?id=ETMEFXMPSS...,den von Clingental,Clingental,0.9525679349899292,rel,Kloster Klingental,1.0,False,47.561705,7.591156
14099,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,d6ab6e15-5a06-4e7f-aade-17399fc78365_20240705,"20, 21",1426,Item git ze kouffende herr Schaler der Ziegler...,https://files.transkribus.eu/Get?id=OAPEIOSDJF...,dem obge frowen ze Klingental,frowen ze Klingental,0.9721857070922851,rel,Kloster Klingental,1.0,False,47.561705,7.591156
14100,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,903b0ee3-2d41-40b0-a25b-753076e3dc1b_20240705,22,1432,Item git ze kouffen Elsy relta quond Antz\nlin...,https://files.transkribus.eu/Get?id=NGZXVNXMCQ...,den frowen ze Klingental,frowen ze Klingental,0.9919654279947281,rel,Kloster Klingental,1.0,False,47.561705,7.591156


# Definiere eine Grundkarte


In [14]:
# Define the basemap.
basemap = WMSLayer(
    url='https://wms.geo.bs.ch/',
    layers='HP_Situationsplan_Basel_1862',
    attribution='Geodaten Kanton Basel-Stadt'
)

# Show the basemap on a map.
m1 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)
m1

Map(center=[47.557, 7.595], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

# Institution in Karte einbauen

In [15]:
# Define a map.
m2 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Add the search results to the map.
for index, row in klingi_df.iterrows():

    # Create Circle.
    circle = Circle(location=(row['lat'], row['lon']),
                    radius=3,
                    color='red',
                    fill_color='red')

    # Add the circle to the map.
    m2.add(circle)

#  Display the map.
m2

Map(center=[47.557, 7.595], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

# Polygon definieren: Kleinbasel

In [16]:


# Add polygon of "Kleinbasel".
polygon_coordinates = [(47.57444952, 7.58560597),
                       (47.57468023, 7.61461808),
                       (47.57468023, 7.61461808),
                       (47.55772285, 7.61381058),
                       (47.55587715, 7.60227495),
                       (47.55619438, 7.59685321),
                       (47.5587899, 7.59154682),
                       (47.56158729, 7.58802845),
                       (47.56663413, 7.58491383),
                       (47.57444952, 7.58560597)]

# Gefilterte Institution auf eine Rheinseite reduzieren (hier Kleinbasel)

In [19]:
from shapely.geometry import Point, Polygon

# Define the polygon
polygon = Polygon(polygon_coordinates)

for index, row in klingi_df.iterrows():

    # Define the point
    point = Point(row['lat'], row['lon'])

    # Check if the point is within the polygon
    klingi_df.at[index, 'in_kleinbasel'] = point.within(polygon)



In [18]:
klingi_df

,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence,fuzzy_match,lat,lon,in_kleinbasel
64,HGB_1_024_040,Barfüsserplatz 15,2611302.138354082,1267114.9650077126,668dfb9d-8780-4d4a-a39b-4be080ecbfe2_20240705,14,1492,Hanns Rosenfeld der Wachtmeister verkauft an L...,https://files.transkribus.eu/Get?id=GTBFRSLYCP...,Clingental,Clingental,0.9173821210861206,rel,Kloster Klingental,1.0,False,47.554654,7.588776,False
159,HGB_1_024_066,Barfüsserplatz 29,2611333.3941644696,1267149.2194807678,ab16bc69-171e-4d91-81da-f43ccb09db9a_20240705,23,1523,"Melchior Lompar der Miner, u. seine Frau Ursul...",https://files.transkribus.eu/Get?id=JQFGPLGQSS...,einer Closterfrauen zu Clingental,Closterfrauen zu Clingental,0.88949915766716,rel,Kloster Klingental,1.0,False,47.554962,7.589192,False
252,HGB_1_024_115,Barfüsserplatz 28,2611327.614136818,1267149.1169899958,801760e2-13f7-4a7b-92ab-b92f55fd325a_20240705,19,1523,Melchior Lumpar der Kaurer u. seine Frau Ursul...,https://files.transkribus.eu/Get?id=BQPFCYPJFZ...,einer Closterfrauen zu Clingenthal,Closterfrauen zu Clingenthal,0.837795689702034,rel,Kloster Klingental,1.0,False,47.554961,7.589115,False
255,HGB_1_024_116,Barfüsserplatz 30,2611334.6649472862,1267157.820265856,5da065f0-d31c-4d3c-a489-04308819cd01_20240705,11,1454,Do gend ze kaufen Mr. Conrat Wagner der Lütenm...,https://files.transkribus.eu/Get?id=IJFCXNSZMK...,die frauen von Clingental,frauen von Clingental,0.9784636348485947,rel,Kloster Klingental,1.0,False,47.555039,7.589209,False
298,HGB_1_058_020,Freie Strasse 1,2611281.9863638678,1267451.4916674045,e5a59ebf-42f3-4e7c-8327-7c8061355f2a_20240705,22,1492,Schultheissen Urkunde.\nFrow Margreth von Buss...,https://files.transkribus.eu/Get?id=XPVYRMZPKN...,den Frowen zu Clingental,Frowen zu Clingental,0.8744285106658936,rel,Kloster Klingental,1.0,False,47.557681,7.588517,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14096,HGB_1_017_008,Badergässlein 3,2611478.9129142133,1267904.6579906673,ddb9dbd0-ab58-4226-b962-95737ec0f4d3_20240705,21,1464,"Frow Else Rocklins, wil. Oswald Stehelin des W...",https://files.transkribus.eu/Get?id=QDXKTZSOSR...,den Frowen zu Clingental,Frowen zu Clingental,0.9514731019735336,rel,Kloster Klingental,1.0,False,47.561754,7.591145,True
14098,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,6dbafa7c-75a6-45de-bb41-5fcad55ba20c_20240705,14,1416,Item het zu koffende zem Gred Frischers den\nh...,https://files.transkribus.eu/Get?id=ETMEFXMPSS...,den von Clingental,Clingental,0.9525679349899292,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True
14099,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,d6ab6e15-5a06-4e7f-aade-17399fc78365_20240705,"20, 21",1426,Item git ze kouffende herr Schaler der Ziegler...,https://files.transkribus.eu/Get?id=OAPEIOSDJF...,dem obge frowen ze Klingental,frowen ze Klingental,0.9721857070922851,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True
14100,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,903b0ee3-2d41-40b0-a25b-753076e3dc1b_20240705,22,1432,Item git ze kouffen Elsy relta quond Antz\nlin...,https://files.transkribus.eu/Get?id=NGZXVNXMCQ...,den frowen ze Klingental,frowen ze Klingental,0.9919654279947281,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True


Neues DF für das Klingental im Kleinbasel

In [23]:
klingi_in_klein_BS = klingi_df[klingi_df["in_kleinbasel"] == True]

In [24]:
klingi_in_klein_BS

,coll_id,coll_house,coll_x,coll_y,doc_id,doc_pageNum,doc_date,doc_fulltext,doc_image,res_full,res_head,ner_confidence,res_cat,res_norm,res_norm_confidence,fuzzy_match,lat,lon,in_kleinbasel
1468,HGB_1_157_013,Rhein und Rheinbrücke: Kleinbasel vor Bläsitho...,2611451.6517668194,1267779.555361138,90418815-d3d9-42fb-a402-d296328498d5_20240705,47,1522,Bob. Conrat Doltter schultheiss ze minderenn B...,https://files.transkribus.eu/Get?id=EORXBMWYNP...,denen von Clingental,denen von Clingental,0.9589317242304484,rel,Kloster Klingental,1.0,False,47.560629,7.590780,True
1473,HGB_1_157_013,Rhein und Rheinbrücke: Kleinbasel vor Bläsitho...,2611451.6517668194,1267779.555361138,664fe3d6-444f-4dc0-9871-a297204ab675_20240705,"50, 51",1522,"Schultheissenurkunde.\nHanns Symon von Oug, se...",https://files.transkribus.eu/Get?id=NUAZSBQKBO...,den von Klingental,Klingental,0.9750435948371887,rel,Kloster Klingental,1.0,False,47.560629,7.590780,True
1476,HGB_1_157_013,Rhein und Rheinbrücke: Kleinbasel vor Bläsitho...,2611451.6517668194,1267779.555361138,62d7268e-8d59-4c59-b1dd-a8b90286c8aa_20240705,"52, 53",1522,Item da gebenn ze kouffenn die ersamen\nmeiste...,https://files.transkribus.eu/Get?id=ZNHNSQDLKN...,denen von Klingenthal,denen von Klingenthal,0.9754649996757507,rel,Kloster Klingental,0.9693205522689193,False,47.560629,7.590780,True
6460,HGB_1_159_025,Rheingasse Th. v. 3 neben 5,2611503.1736354246,1267790.1215433837,fad10da1-37ca-432c-881a-e731b7f04911_20240705,6,1424,Item gebent ze kouffen Eberlin Roßlin der\nhaf...,https://files.transkribus.eu/Get?id=NCKCOEERZZ...,den closterfrowen ze Klingental,closterfrowen ze Klingental,0.954364150762558,rel,Kloster Klingental,1.0,False,47.560723,7.591464,True
6479,HGB_1_159_032,Rheingasse 15,2611526.4759079535,1267760.898269996,3bc3c6c6-dd8b-488e-95a8-4fbc279d9f66_20240705,"17, 18",1463,Item gend zu koffen Peterhans Schaler der\nhaf...,https://files.transkribus.eu/Get?id=PMKMQBCGGK...,minen frowen von Clingental,minen frowen von Clingental,0.9818302094936371,rel,Kloster Klingental,1.0,False,47.560460,7.591773,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14096,HGB_1_017_008,Badergässlein 3,2611478.9129142133,1267904.6579906673,ddb9dbd0-ab58-4226-b962-95737ec0f4d3_20240705,21,1464,"Frow Else Rocklins, wil. Oswald Stehelin des W...",https://files.transkribus.eu/Get?id=QDXKTZSOSR...,den Frowen zu Clingental,Frowen zu Clingental,0.9514731019735336,rel,Kloster Klingental,1.0,False,47.561754,7.591145,True
14098,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,6dbafa7c-75a6-45de-bb41-5fcad55ba20c_20240705,14,1416,Item het zu koffende zem Gred Frischers den\nh...,https://files.transkribus.eu/Get?id=ETMEFXMPSS...,den von Clingental,Clingental,0.9525679349899292,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True
14099,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,d6ab6e15-5a06-4e7f-aade-17399fc78365_20240705,"20, 21",1426,Item git ze kouffende herr Schaler der Ziegler...,https://files.transkribus.eu/Get?id=OAPEIOSDJF...,dem obge frowen ze Klingental,frowen ze Klingental,0.9721857070922851,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True
14100,HGB_1_017_013,Badergässlein 6,2611479.7777292095,1267899.3216729115,903b0ee3-2d41-40b0-a25b-753076e3dc1b_20240705,22,1432,Item git ze kouffen Elsy relta quond Antz\nlin...,https://files.transkribus.eu/Get?id=NGZXVNXMCQ...,den frowen ze Klingental,frowen ze Klingental,0.9919654279947281,rel,Kloster Klingental,1.0,False,47.561705,7.591156,True


# Definiere für die Suchresultate ein Pop-up






In [25]:
# Define a map.
m3 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Add the search results to the map.
for index, row in klingi_in_klein_BS.iterrows():

    # Create Circle.
    circle = Circle(location=(row['lat'], row['lon']),
                    radius=3,
                    color='blue',
                    fill_color='blue')

    # Add pop-up to display attributes of the circle.
    popup_content = f"""
        ID: {row['coll_id']}<br>
        House: {row['coll_house']}<br>
        Pages: {row['doc_pageNum']}<br>
        Year: {row['doc_date']}<br>
        <a href="{row['doc_image']}">Open Image</a>
        """
    popup = Popup(location=(row['lat'], row['lon']), child=widgets.HTML(popup_content), close_button=True)
    circle.popup = popup

    # Add the circle to the map.
    m3.add(circle)

#  Display the map.
m3

Map(center=[47.557, 7.595], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

# Animierte Visualisierung über die Zeit

In [27]:
# Time interval in years.
TIME_INTERVAL = 10

# Time in seconds showing each interval.
TIME_STEP = 5

# Define a map.
m6 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

#  Display the map.
display(m6)

# Determine minimal and maximal year.
min_year = int(klingi_df['doc_date'].min())
max_year = int(klingi_df['doc_date'].max())

# Initialize a group layer.
layer_group = LayerGroup()
m6.add(layer_group)

# Initialize a legend.
legend = LegendControl(legend={})
m6.add(legend)

# Create circles for each time interval.
min_interval = min_year
max_interval = min_interval + TIME_INTERVAL - 1
while min_interval <= max_year:
    # Define the sleeping time.
    time.sleep(TIME_STEP)

    # Define a temporary layer.
    layer_group_new = LayerGroup()

    # Determine the results within the current time interval.
    result_interval = klingi_df[(klingi_df['doc_date'] >= min_interval) & (klingi_df['doc_date'] <= max_interval)]

    for index, row in result_interval.iterrows():
        # Create circle.
        circle = Circle(location=(row['lat'], row['lon']),
                        radius=3,
                        color='blue',
                        fill_color='blue')

        # Add circle to the group layer.
        layer_group_new.add(circle)

    # Update the group layer.
    m6.remove(layer_group)
    layer_group = layer_group_new
    m6.add(layer_group)

    # Update the legend.
    m6.remove(legend)
    legend = LegendControl(legend={f'{min_interval} - {min(max_interval, max_year)}': 'blue'})
    m6.add(legend)

    # Update the interval.
    min_interval += TIME_INTERVAL
    max_interval += TIME_INTERVAL

Map(center=[47.557, 7.595], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### Interval 33 Years

In [29]:
# Time interval in years.
TIME_INTERVAL = 33

# Time in seconds showing each interval.
TIME_STEP = 3

# Define a map.
m6 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

#  Display the map.
display(m6)

# Determine minimal and maximal year.
min_year = int(klingi_df['doc_date'].min())
max_year = int(klingi_df['doc_date'].max())

# Initialize a group layer.
layer_group = LayerGroup()
m6.add(layer_group)

# Initialize a legend.
legend = LegendControl(legend={})
m6.add(legend)

# Create circles for each time interval.
min_interval = min_year
max_interval = min_interval + TIME_INTERVAL - 1
while min_interval <= max_year:
    # Define the sleeping time.
    time.sleep(TIME_STEP)

    # Define a temporary layer.
    layer_group_new = LayerGroup()

    # Determine the results within the current time interval.
    result_interval = klingi_df[(klingi_df['doc_date'] >= min_interval) & (klingi_df['doc_date'] <= max_interval)]

    for index, row in result_interval.iterrows():
        # Create circle.
        circle = Circle(location=(row['lat'], row['lon']),
                        radius=3,
                        color='blue',
                        fill_color='blue')

        # Add circle to the group layer.
        layer_group_new.add(circle)

    # Update the group layer.
    m6.remove(layer_group)
    layer_group = layer_group_new
    m6.add(layer_group)

    # Update the legend.
    m6.remove(legend)
    legend = LegendControl(legend={f'{min_interval} - {min(max_interval, max_year)}': 'blue'})
    m6.add(legend)

    # Update the interval.
    min_interval += TIME_INTERVAL
    max_interval += TIME_INTERVAL

Map(center=[47.557, 7.595], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…